In [6]:
#!pip install -U pytest

The below code verifies that all packages installed in your Python environment (e.g., a Conda environment) are importable and function as expected. It serves as a basic "smoke test" by checking that each package can be imported and either exposes a __version__ attribute or provides a test/example function

#### Features
Dynamic Import Validation:
Iterates through all installed packages and attempts to import them.

Basic Functionality Check:
If a package has a __version__ attribute, it reports the version. If not, it attempts to run a built-in test or example function if available.

Pytest Integration:
Includes a pytest-style test (test_installed_packages) that will fail with detailed error messages if any package fails verification.

Fallback for Non-Pytest Environments:
Provides a fallback implementation to raise assertions when pytest is not installed.

#### How It Works
Module Name Mapping & Resolution:
The get_module_name() function handles cases where a package’s import name differs from its distribution name (e.g., "Pillow" is imported as "PIL") or simply replaces hyphens with underscores.

Package Verification:
The verify_package() function attempts to import the module and checks for a __version__ attribute. If not present, it tries to run a test or example function (if available) and returns a tuple containing the version (or "N/A") and a log message.

CSV Output:
The output_results_to_csv() function converts the list of results into a pandas DataFrame, sorts it by package name, and writes it to a CSV file (default name is "package_verification.csv").

Pytest Integration:
The test_installed_packages() function is formatted for pytest. When you run pytest in your project directory, it will automatically execute and report any errors in package verification.

Standalone Execution:
Running the script directly with python your_script.py prints package verification results to the terminal and outputs a CSV file with the results.

In [ ]:
import pkg_resources
import importlib
import traceback
import pandas as pd

# Attempt to import pytest for unit testing; if not available, define a fallback.
try:
    import pytest
except ModuleNotFoundError:
    print("pytest module not found. Using a fallback for unit testing assertions.")
    class PytestFallback:
        @staticmethod
        def fail(msg):
            raise AssertionError(msg)
    pytest = PytestFallback()

# Mapping for packages where the importable module name differs from the package name.
MODULE_NAME_MAPPING = {
    'Pillow': 'PIL',
    'scikit-learn': 'sklearn',
    'PyYAML': 'yaml',
    # Add more mappings here as needed.
}

def get_module_name(package_name):
    """
    Returns the correct module name for a given package.
    Checks MODULE_NAME_MAPPING; if not found, replaces hyphens with underscores.
    """
    if package_name in MODULE_NAME_MAPPING:
        return MODULE_NAME_MAPPING[package_name]
    return package_name.replace("-", "_")

def verify_package(package_name):
    """
    Attempts to import and test a single package.
    
    Returns:
        tuple: (version, log_message)
          - version: The __version__ attribute if available, "N/A" if not, or None if an error occurs.
          - log_message: A message detailing the outcome.
    """
    module_name = get_module_name(package_name)
    try:
        module = importlib.import_module(module_name)
    except Exception as e:
        err = f"❌ Failed to import module '{module_name}': {e}"
        return None, err
    
    if hasattr(module, '__version__'):
        version = getattr(module, '__version__')
        log = f"✅ Imported successfully. Version: {version}"
        return version, log
    elif hasattr(module, 'test') or hasattr(module, 'example'):
        try:
            if hasattr(module, 'test'):
                module.test()
            elif hasattr(module, 'example'):
                module.example()
            version = "N/A"
            log = "✅ Imported successfully, and test/example executed."
            return version, log
        except Exception as e:
            err = f"❌ Imported but test/example failed: {e}"
            return None, err
    else:
        version = "N/A"
        log = "✅ Imported successfully (no version or test function available)."
        return version, log

def output_results_to_csv(results, filename="package_verification.csv"):
    """
    Exports the verification results to a CSV file with three columns:
      Package Name, Version, Log Message.
    """
    df = pd.DataFrame(results, columns=["Package Name", "Version", "Log Message"])
    df.sort_values(by="Package Name", inplace=True)
    df.to_csv(filename, index=False)
    print(f"\nCSV file written to: {filename}")

def test_installed_packages():
    """
    PyTest test function that verifies the import and basic functionality
    of each installed package. It collects errors and will fail if any occur.
    """
    errors = []
    for dist in pkg_resources.working_set:
        package_name = dist.project_name
        module_name = get_module_name(package_name)
        try:
            version, log = verify_package(package_name)
            # Assert that a valid log message is returned.
            assert log, f"Empty result for package {package_name}"
            print(f"✅ {package_name} imported as '{module_name}' with result: {log}")
        except Exception as e:
            errors.append(f"{package_name} (as '{module_name}') failed: {e}")
    
    if errors:
        error_message = "\n".join(errors)
        pytest.fail(f"Some packages failed verification:\n{error_message}")

def main():
    """
    Main function to verify all installed packages and output the results to a CSV file.
    """
    print("🔍 Verifying installed packages...\n")
    results = []  # List to hold tuples of (package name, version, log message)
    
    for dist in pkg_resources.working_set:
        package_name = dist.project_name
        version, log_message = verify_package(package_name)
        results.append((package_name, version if version is not None else "", log_message))
        print(f"{package_name}: {log_message}")
    
    # Output the results to a CSV file
    output_results_to_csv(results)

if __name__ == "__main__":
    main()
